<a href="https://colab.research.google.com/github/kafilatmusa1-droid/hausa-topic-classification/blob/main/HausaTopic_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib xgboost model2vec gradio bertopic

In [ ]:
import sklearn, xgboost, model2vec, gradio, bertopic
print("All packages loaded")

All packages loaded


In [ ]:
!pip install -q git+https://github.com/ZiyaoLi/fast-kan.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from fastkan import FastKAN
print("FastKAN loaded")

FastKAN loaded


In [ ]:
from datasets import load_dataset
ds = load_dataset("masakhane/masakhanews", "hau")
print(ds)
print(ds["train"].column_names)
print(ds["train"].features["label"].names)

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

train.tsv:   0%|          | 0.00/5.58M [00:00<?, ?B/s]

dev.tsv:   0%|          | 0.00/770k [00:00<?, ?B/s]

test.tsv:   0%|          | 0.00/1.71M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2219 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/317 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/637 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['category', 'headline', 'text', 'url'],
        num_rows: 2219
    })
    validation: Dataset({
        features: ['category', 'headline', 'text', 'url'],
        num_rows: 317
    })
    test: Dataset({
        features: ['category', 'headline', 'text', 'url'],
        num_rows: 637
    })
})
['category', 'headline', 'text', 'url']


KeyError: 'label'

In [ ]:
import collections
print(collections.Counter(ds["train"]["category"]))

Counter({'entertainment': 350, 'politics': 350, 'sports': 347, 'religion': 345, 'health': 345, 'business': 279, 'technology': 203})


In [ ]:
label_names = sorted(set(ds["train"]["category"]))
label2id = {name: i for i, name in enumerate(label_names)}

def prepare(example):
    example["input"] = example["headline"] + " " + example["text"]
    example["label"] = label2id[example["category"]]
    return example

ds = ds.map(prepare)

train_x = ds["train"]["input"]
train_y = ds["train"]["label"]
val_x = ds["validation"]["input"]
val_y = ds["validation"]["label"]
test_x = ds["test"]["input"]
test_y = ds["test"]["label"]
n_classes = len(label_names)

print(label_names)
print(len(train_x), len(val_x), len(test_x))

Map:   0%|          | 0/2219 [00:00<?, ? examples/s]

Map:   0%|          | 0/317 [00:00<?, ? examples/s]

Map:   0%|          | 0/637 [00:00<?, ? examples/s]

['business', 'entertainment', 'health', 'politics', 'religion', 'sports', 'technology']
2219 317 637


In [ ]:
from sklearn.metrics import f1_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

def weighted_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average="weighted")

vec = CountVectorizer()
Xtr = vec.fit_transform(train_x)
Xte = vec.transform(test_x)

baseline = {}
for name in ["NaiveBayes", "MLP", "XGBoost"]:
    scores = []
    for seed in range(5):
        if name == "NaiveBayes":
            model = MultinomialNB()
        elif name == "MLP":
            model = MLPClassifier(max_iter=200, random_state=seed)
        else:
            model = XGBClassifier(random_state=seed)
        model.fit(Xtr, train_y)
        scores.append(weighted_f1(test_y, model.predict(Xte)))
    baseline[name] = sum(scores) / 5
    print(name, round(baseline[name] * 100, 1))

NaiveBayes 87.2
MLP 88.5
XGBoost 83.7


In [ ]:
train_y = list(train_y)
test_y = list(test_y)
print(type(train_y), len(train_y))

<class 'list'> 2219


In [ ]:
import numpy as np
import scipy.sparse as sp
from sklearn.preprocessing import normalize
from bertopic.vectorizers import ClassTfidfTransformer

cv = CountVectorizer(max_features=5000)
Xc_train = cv.fit_transform(train_x)
Xc_test = cv.transform(test_x)

class_docs = []
for c in range(n_classes):
    docs = [train_x[i] for i in range(len(train_x)) if train_y[i] == c]
    class_docs.append(" ".join(docs))
Xclass = cv.transform(class_docs)

W = ClassTfidfTransformer().fit_transform(Xclass)
w = np.asarray(W.mean(axis=0)).ravel()
D = sp.diags(w)

ctf_train = normalize(Xc_train @ D)
ctf_test = normalize(Xc_test @ D)
print(ctf_train.shape, ctf_test.shape)

(2219, 5000) (637, 5000)


In [ ]:
from model2vec.distill import distill

m2v = distill("sentence-transformers/LaBSE", pca_dims=256)
m2v_train = m2v.encode(train_x)
m2v_test = m2v.encode(test_x)
print(m2v_train.shape, m2v_test.shape)

config.json:   0%|          | 0.00/804 [00:00<?, ?B/s]

[ERROR] `cache_position` is part of BertModel.forward's signature, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/bert/modeling_bert.py.
[ERROR] `cache_position` is part of BertLMHeadModel.forward's signature, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/bert/modeling_bert.py.


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.88GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/5.22M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.62M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Encoding tokens: 100%|██████████| 501051/501051 [56:48<00:00, 146.98 tokens/s]


(2219, 256) (637, 256)


In [ ]:
!pip install -q "model2vec[distill]"

In [ ]:
from model2vec import StaticModel

pot = StaticModel.from_pretrained("minishlab/potion-multilingual-128M")
pot_train = pot.encode(train_x)
pot_test = pot.encode(test_x)
print(pot_train.shape, pot_test.shape)

(2219, 256) (637, 256)


In [ ]:
from datasets import load_dataset

ds = load_dataset("masakhane/masakhanews", "hau")
label_names = sorted(set(ds["train"]["category"]))
label2id = {name: i for i, name in enumerate(label_names)}

def prepare(example):
    example["input"] = example["headline"] + " " + example["text"]
    example["label"] = label2id[example["category"]]
    return example

ds = ds.map(prepare)

train_x = ds["train"]["input"]
train_y = list(ds["train"]["label"])
val_x = ds["validation"]["input"]
val_y = list(ds["validation"]["label"])
test_x = ds["test"]["input"]
test_y = list(ds["test"]["label"])
n_classes = len(label_names)

print(len(train_x), len(val_x), len(test_x), n_classes)

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

train.tsv:   0%|          | 0.00/5.58M [00:00<?, ?B/s]

dev.tsv:   0%|          | 0.00/770k [00:00<?, ?B/s]

test.tsv:   0%|          | 0.00/1.71M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2219 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/317 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/637 [00:00<?, ? examples/s]

Map:   0%|          | 0/2219 [00:00<?, ? examples/s]

Map:   0%|          | 0/317 [00:00<?, ? examples/s]

Map:   0%|          | 0/637 [00:00<?, ? examples/s]

2219 317 637 7


In [ ]:
import numpy as np
import scipy.sparse as sp

np.save("pot_train.npy", pot_train)
np.save("pot_test.npy", pot_test)
sp.save_npz("ctf_train.npz", ctf_train)
sp.save_npz("ctf_test.npz", ctf_test)
print("saved")

NameError: name 'ctf_train' is not defined

In [ ]:
import numpy as np
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import normalize
from bertopic.vectorizers import ClassTfidfTransformer

cv = CountVectorizer(max_features=5000)
Xc_train = cv.fit_transform(train_x)
Xc_test = cv.transform(test_x)

class_docs = []
for c in range(n_classes):
    docs = [train_x[i] for i in range(len(train_x)) if train_y[i] == c]
    class_docs.append(" ".join(docs))
Xclass = cv.transform(class_docs)

W = ClassTfidfTransformer().fit_transform(Xclass)
w = np.asarray(W.mean(axis=0)).ravel()
D = sp.diags(w)

ctf_train = normalize(Xc_train @ D)
ctf_test = normalize(Xc_test @ D)

sp.save_npz("ctf_train.npz", ctf_train)
sp.save_npz("ctf_test.npz", ctf_test)
np.save("pot_train.npy", pot_train)
np.save("pot_test.npy", pot_test)
print("saved", ctf_train.shape, ctf_test.shape)

saved (2219, 5000) (637, 5000)


In [ ]:
!pip install -q bertopic

In [ ]:
from sklearn.metrics import f1_score
from sklearn.svm import LinearSVC

def weighted_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average="weighted")

results = {}
encoders = {
    "c-TF-IDF": (ctf_train, ctf_test),
    "Potion": (pot_train, pot_test),
}
for enc_name, (Xtr_e, Xte_e) in encoders.items():
    model = LinearSVC()
    model.fit(Xtr_e, train_y)
    pred = model.predict(Xte_e)
    results[enc_name + " + SVM"] = weighted_f1(test_y, pred)

for k, v in results.items():
    print(k, round(v * 100, 1))

c-TF-IDF + SVM 55.5
Potion + SVM 85.7


In [ ]:
import torch
import torch.nn as nn
from fastkan import FastKAN

def run_fastkan(Xtr, Xte, seed=0, epochs=30):
    torch.manual_seed(seed)
    Xtr_t = torch.tensor(Xtr, dtype=torch.float32)
    Xte_t = torch.tensor(Xte, dtype=torch.float32)
    ytr_t = torch.tensor(train_y, dtype=torch.long)
    model = FastKAN(layers_hidden=[Xtr_t.shape[1], 64, n_classes])
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(Xtr_t))
        for i in range(0, len(perm), 32):
            idx = perm[i:i + 32]
            opt.zero_grad()
            loss = loss_fn(model(Xtr_t[idx]), ytr_t[idx])
            loss.backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        return model(Xte_t).argmax(dim=1).numpy()

fk_scores = []
for seed in range(5):
    pred = run_fastkan(pot_train, pot_test, seed=seed)
    fk_scores.append(weighted_f1(test_y, pred))
results["Potion + FastKAN"] = sum(fk_scores) / 5
print("Potion + FastKAN", round(results["Potion + FastKAN"] * 100, 1))

Potion + FastKAN 82.7


In [ ]:
!pip install -q git+https://github.com/ZiyaoLi/fast-kan.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
ctf_train_d = ctf_train.toarray()
ctf_test_d = ctf_test.toarray()

fk_scores = []
for seed in range(5):
    pred = run_fastkan(ctf_train_d, ctf_test_d, seed=seed)
    fk_scores.append(weighted_f1(test_y, pred))
results["c-TF-IDF + FastKAN"] = sum(fk_scores) / 5
print("c-TF-IDF + FastKAN", round(results["c-TF-IDF + FastKAN"] * 100, 1))

c-TF-IDF + FastKAN 65.3


In [ ]:
import gradio as gr
from sklearn.svm import LinearSVC

final_clf = LinearSVC()
final_clf.fit(pot_train, train_y)

def predict_topic(text):
    vector = pot.encode([text])
    label_id = final_clf.predict(vector)[0]
    return label_names[label_id]

demo = gr.Interface(
    fn=predict_topic,
    inputs=gr.Textbox(lines=5, label="Paste Hausa news text"),
    outputs=gr.Textbox(label="Predicted topic"),
    title="Hausa News Topic Classifier",
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8b698bea308544d46c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
for i in [0, 1, 2]:
    print(test_x[i][:300])
    print("CORRECT TOPIC:", label_names[test_y[i]])
    print("---")

Najeriya: Kayan gwari sun yi kwantai a arewa bayan hana kai kaya kudu A yayin da farashin kayan gwari da dabbobi suka tashi a kudancin Najeriya,  a arewacin kuma farashin kayayyakin ne ya karye. Tasirin yajin aikin Haɗaɗɗiyar ƙungiyar masu kayan gwari da dabbobi da kayan cimaka ya haifar da tashin f
CORRECT TOPIC: business
---
Abin da ya kamata ku sani game da kama gwamnan CBN Emefiele Rahotannin da ke kara-kaina a wasu kafafen yada labarai na cewa hukumar tsaron farin kaya na shirin kama gwamnan babban bankin Najeriya CBN, Godwin Emefiele. Wata babbar kotu a Abuja babban birnin kasar ta yi watsi da bukatar DSS ta kamawa 
CORRECT TOPIC: business
---
Elisa Loncón: Matar da ta samu ɗaukakar rubuta tsarin mulkin ƙasa bayan fama da talauci A shekarun 1970 lokacin da Elisa Loncón ke tasowa, sai ta yi tafiyar mil 8 kafin ta isa makaranta. Iyayenta 'yan asalin yankin Mapuche ne na kasar Chile, wadanda ke rayuwa cikin bakin talauci a wani kauye da ke k
CORRECT TOPIC: business
---


In [ ]:
import pandas as pd

scores = {
    "Naive Bayes (replication)": 87.2,
    "MLP (replication)": 88.5,
    "XGBoost (replication)": 83.7,
    "c-TF-IDF + SVM": 55.5,
    "c-TF-IDF + FastKAN": 65.3,
    "Potion + SVM": 85.7,
    "Potion + FastKAN": 82.7,
}
df = pd.DataFrame(list(scores.items()), columns=["model", "weighted_f1"])
df.to_csv("results.csv", index=False)
print(df)

                       model  weighted_f1
0  Naive Bayes (replication)         87.2
1          MLP (replication)         88.5
2      XGBoost (replication)         83.7
3             c-TF-IDF + SVM         55.5
4         c-TF-IDF + FastKAN         65.3
5               Potion + SVM         85.7
6           Potion + FastKAN         82.7


In [ ]:
!pip install -q "model2vec[distill]"

In [20]:
from model2vec.distill import distill
import numpy as np

m2v = distill("sentence-transformers/LaBSE", pca_dims=256)
m2v_train = m2v.encode(train_x)
m2v_test = m2v.encode(test_x)
np.save("m2v_train.npy", m2v_train)
np.save("m2v_test.npy", m2v_test)
print(m2v_train.shape, m2v_test.shape)

config.json:   0%|          | 0.00/804 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.88GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/5.22M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.62M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Encoding tokens: 100%|██████████| 501051/501051 [1:34:00<00:00, 88.83 tokens/s]


(2219, 256) (637, 256)


In [21]:
import numpy as np
m2v_train = np.load("m2v_train.npy")
m2v_test = np.load("m2v_test.npy")

In [22]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import f1_score
from sklearn.svm import LinearSVC
from fastkan import FastKAN

m2v_train = np.load("m2v_train.npy")
m2v_test = np.load("m2v_test.npy")

try:
    results
except NameError:
    results = {}

def weighted_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average="weighted")

def run_fastkan(Xtr, Xte, seed=0, epochs=30):
    torch.manual_seed(seed)
    Xtr_t = torch.tensor(Xtr, dtype=torch.float32)
    Xte_t = torch.tensor(Xte, dtype=torch.float32)
    ytr_t = torch.tensor(train_y, dtype=torch.long)
    model = FastKAN(layers_hidden=[Xtr_t.shape[1], 64, n_classes])
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(Xtr_t))
        for i in range(0, len(perm), 32):
            idx = perm[i:i + 32]
            opt.zero_grad()
            loss = loss_fn(model(Xtr_t[idx]), ytr_t[idx])
            loss.backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        return model(Xte_t).argmax(dim=1).numpy()

svm = LinearSVC()
svm.fit(m2v_train, train_y)
results["Model2Vec + SVM"] = weighted_f1(test_y, svm.predict(m2v_test))

fk_scores = []
for seed in range(5):
    pred = run_fastkan(m2v_train, m2v_test, seed=seed)
    fk_scores.append(weighted_f1(test_y, pred))
results["Model2Vec + FastKAN"] = sum(fk_scores) / 5

print("Model2Vec + SVM", round(results["Model2Vec + SVM"] * 100, 1))
print("Model2Vec + FastKAN", round(results["Model2Vec + FastKAN"] * 100, 1))

Model2Vec + SVM 84.0
Model2Vec + FastKAN 82.9


In [23]:
import pandas as pd

df = pd.read_csv("results.csv")
df.loc[len(df)] = ["Model2Vec + SVM", results["Model2Vec + SVM"] * 100]
df.loc[len(df)] = ["Model2Vec + FastKAN", results["Model2Vec + FastKAN"] * 100]
df.to_csv("results.csv", index=False)
print(df)

                       model  weighted_f1
0  Naive Bayes (replication)    87.200000
1          MLP (replication)    88.500000
2      XGBoost (replication)    83.700000
3             c-TF-IDF + SVM    55.500000
4         c-TF-IDF + FastKAN    65.300000
5               Potion + SVM    85.700000
6           Potion + FastKAN    82.700000
7            Model2Vec + SVM    84.040288
8        Model2Vec + FastKAN    82.934060
